# Projektaufgabe "Mobile Roboter"

Teilnehmer:
- Martin Filpe ubpmo@student.kit.edu
- Luca Beck ugkld@student.kit.edu
- Christian Diehm ufrnu@student.kit.edu

### I. Implementieren Sie einen CollisionChecker, der es erlaubt, einen holonomen „mobilen“ Roboter mit 2 (x, y) bzw. 3 Freiheitsgraden (x, y, rot z) und Hindernissen mit den Planungsverfahren aus der Vorlesung zu planen, ohne dass die Planungsverfahren verändert werden müssen.

### a) Erläutern Sie, was „holonom“ in diesem Kontext bedeutet.

In diesem Kontext bedeutet „holonom“, dass der mobile Roboter keine kinematischen Zwangsbedingungen hat, die seine Bewegungsrichtung einschränken. Er kann sich frei in der Ebene bewegen, also unabhängig in x- und y-Richtung sowie ggf. um die z-Achse rotieren (bei 3 DOF). Das heißt, er kann direkt jede beliebige Position und Orientierung anfahren, ohne Einschränkungen wie bei einem Auto, das z. B. nicht seitwärts fahren kann.

In [ ]:
# Importieren der Bibliotheken
import random
import math
import pickle
import os
import time

import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from tqdm.notebook import tqdm
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML, display

from matplotlib.patches import Patch
from matplotlib import animation, rcParams

from shapely.geometry import box
from shapely.affinity import rotate, translate

from HelperFunctions import *
from MultiRobotHelperFunctions import *

# Importieren der Test-Suite, Benchmark Templates und Kollisionsprüfern
from dependencies.IPTestSuiteMR import benchList, scenes, robots, create_random_benchmark
from dependencies.IPBenchmark import Benchmark
from IPMobileRobotCollisionChecker import MobileRobotCollisionChecker
from IPMultiMobileRobotCollisionChecker import MultiMobileRobotCollisionChecker

# Importieren der Planungsalgorithmen
from dependencies.IPBasicPRM import BasicPRM
from IPAdaptedBasicPRM import AdaptedBasicPRM
from dependencies.IPLazyPRM import LazyPRM
from dependencies.IPVisibilityPRM import VisPRM, VisibilityStatsHandler
from dependencies.IPRRT import RRTSimple

### b) Ermöglichen Sie es in einer Szene einen Roboter (und Hindernissen) mit beliebiger Geometrie zu erzeugen.

In [ ]:
RbenchList = []
# Zufälligen Benchmark erstellen
random_benchmark = create_random_benchmark(name_prefix="testscene", dof=3)
RbenchList.append(random_benchmark)

benchmark = RbenchList[0]
checker = benchmark.collisionChecker
scene = checker.scene
robot_shape = checker.robot_shape
start = benchmark.startList[0]
goal = benchmark.goalList[0]
dof = len(start)

# Roboter transformieren
robot_start = transform_robot(start, robot_shape)
robot_goal = transform_robot(goal, robot_shape)

# Plot erstellen
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_aspect('equal')
ax.set_xlim(checker.limits[0])
ax.set_ylim(checker.limits[1])
ax.set_title(f"Szene ({benchmark.name})")

# Hindernisse zeichnen
plot_geometries(ax, scene.values(), facecolor='lightgray', edgecolor='black', alpha=0.7)

# Roboterpositionen zeichnen
plot_geometries(ax, [robot_start], facecolor='green', edgecolor='black', alpha=0.6)
plot_geometries(ax, [robot_goal], facecolor='blue', edgecolor='black', alpha=0.6)

legend_elements = [
    Patch(facecolor='green', edgecolor='black', label='Startposition'),
    Patch(facecolor='blue', edgecolor='black', label='Zielposition')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.show()

### c) Erzeugen Sie den MobileRobotCollision Checker. 

Test des Colision Checkers

In [ ]:
# Setup
robot_shape = box(-1, -0.5, 1, 0.5)
scene = {"obstacle1": box(5,5,8,8), "obstacle2": box(10,2,12,4)}
checker = MobileRobotCollisionChecker(robot_shape, scene, limits=[[0,20],[0,20],[0,360]])
tests = [([6,5,0], "Test 1"), ([4.6,4,45], "Test 2"), ([11,8,90], "Test 3")]

# Aufruf
plot_collision_tests(robot_shape, scene, checker, tests)


Dynamischer Test des Colision Checkers

In [ ]:
# Kann 1-2 Minuten dauern für das Rendern der Animation
animate_robot_scene()


### d) Evaluieren Sie BasicPRM, LazyPRM, VisibilityPRM und RRT anhand von mindestens 5 Benchmarkumgebungen und jeweils unterschiedlichen Robotertypen (3x2-DoF, 3x3-DoF)

#### a. Erzeugen Sie 5 Benchmarks, in denen sie drei 2-Dof Roboter und drei 3-Dof- Roboter fahren lassen (Scheibenroboter oder Symmetrische Roboter zählen nicht). Die Aufgabenstellungen sollen herausfordernd sein, entweder durch Form des Roboters oder durch Platzierung und Form der Hindernisse.

Alle 30 Benchmarks:

In [ ]:
# Benchmark-Anzahl und Layout
n = len(benchList)
cols = 6
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
axes = axes.flatten()

for idx, benchmark in enumerate(benchList):
    ax = axes[idx]
    checker = benchmark.collisionChecker
    scene = checker.scene
    robot_shape = checker.robot_shape
    limits = checker.limits
    start = benchmark.startList[0]
    goal = benchmark.goalList[0]

    ax.set_xlim(*limits[0])
    ax.set_ylim(*limits[1])
    ax.set_aspect("equal")
    ax.set_title(f"Benchmark {idx + 1}", fontsize=9)

    # Hindernisse
    for obs in scene.values():
        if hasattr(obs, 'exterior'):
            x, y = obs.exterior.xy
            ax.fill(x, y, color="red", alpha=0.5)

    # Startposition
    if len(start) == 3:
        r_start = translate(rotate(robot_shape, start[2], origin="centroid"), xoff=start[0], yoff=start[1])
    else:
        r_start = translate(robot_shape, xoff=start[0], yoff=start[1])
    ax.fill(*r_start.exterior.xy, color="green", alpha=0.6)
    ax.text(start[0], start[1], "Start", ha="center", va="center", fontsize=7)

    # Zielposition
    if len(goal) == 3:
        r_goal = translate(rotate(robot_shape, goal[2], origin="centroid"), xoff=goal[0], yoff=goal[1])
    else:
        r_goal = translate(robot_shape, xoff=goal[0], yoff=goal[1])
    ax.fill(*r_goal.exterior.xy, color="blue", alpha=0.4)
    ax.text(goal[0], goal[1], "Ziel", ha="center", va="center", fontsize=7)

for i in range(n, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()



#### b. Visualisieren Sie den Konfigurationsraum 2-DOF und 3-DoF für die Benchmarkumgebungen und die jeweiligen Roboter. 
#### c. Stellen Sie die den Lösungspfad und die Bewegung des Roboters grafisch dar und erzeugen Sie eine Animation, für Arbeits- und Konfigurationsraum, wie der mobile Roboter entlang des Lösungspfades fährt.


In [ ]:
mpl.rcParams['animation.embed_limit'] = 200
max_attempts = 10

BasicPRM

In [ ]:
basic_runner = PlannerRunner(BasicPRM, {'radius': 10.0, 'numNodes': 300}, name="BasicPRM")
basic_runner.run_benchmarks(
    [benchList[0]], max_attempts=5,
    fps=50, steps_per_segment=20,
    save_animation=False, animation_dir="animationen"
)

LazyPRM

In [ ]:
basic_runner = PlannerRunner(LazyPRM, {'initialRoadmapSize': 200, 'updateRoadmapSize': 50, 'kNearest': 10, 'maxIterations': 50}, name="LazyPRM")
basic_runner.run_benchmarks(
    [benchList[0]], max_attempts=5,
    fps=50, steps_per_segment=20,
    save_animation=False, animation_dir="animationen"
)

VisibilityPRM

In [ ]:
basic_runner = PlannerRunner(VisPRM, {'ntry': 50}, name="VisPRM")
basic_runner.run_benchmarks(
    [benchList[11]], max_attempts=5,
    fps=50, steps_per_segment=20,
    save_animation=False, animation_dir="animationen"
)

RRT

In [ ]:
basic_runner = PlannerRunner(RRTSimple, {'numberOfGeneratedNodes': 300, 'testGoalAfterNumberOfNodes': 10}, name="RRTSimple")
basic_runner.run_benchmarks(
    [benchList[11]], max_attempts=5,
    fps=50, steps_per_segment=20,
    save_animation=False, animation_dir="animationen"
)


#### d. Vergleichen Sie die Planungsverfahren hinsichtlich Suchzeit, Größe Roadmap, Anzahl Punkte im Pfad, Länge des Pfades. 

In [ ]:
skip_all_benchmarks = True
def run_all_benchmarks():
    # -------------------------------
    # Konfigurationen
    # -------------------------------

    # BasicPRM
    config_basic = {
        'radius': 5.0,
        'numNodes': 300
    }

    # LazyPRM
    config_lazy = {
        'initialRoadmapSize': 300,
        'updateRoadmapSize': 50,
        'kNearest': 10,
        'maxIterations': 40
    }

    # RRT
    config_rrt = {
        'numberOfGeneratedNodes': 300,
        'testGoalAfterNumberOfNodes': 10
    }

    # VisPRM
    config_vis = {
        'ntry': 40  
    }

    # -------------------------------
    # Benchmark-Durchläufe
    # -------------------------------

    bench_range_basic = benchList
    bench_range_lazy = benchList
    bench_range_rrt = benchList
    bench_range_vis = benchList 

    results = []

    # BasicPRM
    results += run_benchmark_adaptive_multi_try_sampling(
        BasicPRM, "BasicPRM", config_basic, bench_range_basic,
        max_attempts=5, scale_factor=1.5, max_scalings=5
    )

    # LazyPRM
    results += run_benchmark_adaptive_multi_try_sampling(
        LazyPRM, "LazyPRM", config_lazy, bench_range_lazy,
        max_attempts=5, scale_factor=1.5, max_scalings=5
    )

    # VisPRM
    results += run_benchmark_adaptive_multi_try_sampling(
        VisPRM, "VisPRM", config_vis, bench_range_vis,
        max_attempts=5, scale_factor=1.5, max_scalings=5
    )

    # RRTSimple
    results += run_benchmark_adaptive_multi_try_sampling(
        RRTSimple, "RRTSimple", config_rrt, bench_range_rrt,
        max_attempts=5, scale_factor=1.5, max_scalings=5
    )

    # -------------------------------
    # Ergebnisse speichern
    # -------------------------------

    from IPython.display import display
    import pickle

    with open('results/benchmark_results.pkl', 'wb') as f:
        pickle.dump(results, f)

    df_results = pd.DataFrame(results)
    df_results = df_results[['Planner', 'Benchmark', 'Time [s]', 'Roadmap Size', 'Path Points', 'Path Length']]
    df_results.to_csv('results/benchmark_results.csv', index=False)

    display(df_results)
    
if not skip_all_benchmarks:
    run_all_benchmarks()


##### Zeigt die Parameter pro Benchmark an

In [ ]:
visualize_params_custom_layout('results/found_params.json')

#### Tabelarische und grafische darstellung der Mittelwerte der Vergleiche

In [ ]:
plot_benchmark_summary("results/benchmark_results.csv")

#### Animation ausgewählter Benchmarks

In [ ]:
# Kann je nach Benchmark mehrere Minuten dauern

skip_animation = False
if not skip_animation:
    generate_animations(
        results_file="results/benchmark_results.pkl",
        benchmark_idx=0,
        planners=["BasicPRM", "LazyPRM", "VisPRM", "RRTSimple"],
        save_dir="animationen",
        fps=20,
        steps_per_segment=1,
        save=False,
        all_planners=False,
        selected_idx=3
    )

#### Darstellung der Suchzeit, Roadmap Größe, Punkte im Pfad und der Pfadlänge für jeden Benchmark

In [ ]:
plot_benchmark_metrics("results/benchmark_results.csv")

# Planung mehrerer mobiler Roboter

### e) Erzeugung der Benchmarks und des Kollisionsprüfers.

Im nun folgenden Abschnitt werden mehrere mobile Roboter innerhalb der gleichen Benchmark-Umgebung geplant. Hierfür wird ein neuer Kollisionsprüfer ```IPMultiMobileRobotCollisionChecker.py``` verwendet, der die Kollisionsprüfung für mehrere Roboter implementiert und hierbei auch Kollisionen zwischen den Robotern berücksichtigt.

Zur Auswertung werden zehn manuell erstellte Benchmarks verwendet. Jede der fünf Szenen enthält zunächst zwei, anschließend drei Roboter.
Es gibt ebenfalls die Möglichkeit, die Benchmarks manuell erstellen zu lassen und hierbei die Anzahl der Roboter festzulegen.

In [ ]:
# multiRobotBenchList = build_random_benchmarks(num_robots=2)
multiRobotBenchList = get_predefined_benchmarks()

visualize_multi_robot_benchmarks(multiRobotBenchList)

# Zur Darstellung von größeren Animationen muss der Parameter `animation.embed_limit` von Matplotlib erhöht werden:
mpl.rcParams['animation.embed_limit'] = 200

### b. Visualisieren Sie den Konfigurationsraum 2-DOF und 3-DoF für die Benchmarkumgebungen und die jeweiligen Roboter. 
### c. Stellen Sie die den Lösungspfad und die Bewegung des Roboters grafisch dar und erzeugen Sie eine Animation, für Arbeits- und Konfigurationsraum, wie die mobilen Roboter fahren.

Zur Optimierung der Parameter für die beiden Verfahren BasicPRM und LazyPRM verwenden wir die Berechnung von [Karaman et al.](https://arxiv.org/abs/1105.1186).

Werden die Parameter `radius` für BasicPRM sowie `kNearest` für LazyPRM weg gelassen, so werden sie im `MultiRobotPlannerRunner` optimiert. Alternativ können auch feste Werte übermittelt werden.

#### BasicPRM

In [ ]:
basic_prm_config = {'numNodes': 500} # Radius wird im MultiRobotPlannerRunner berechnet

basic_runner = MultiRobotPlannerRunner(AdaptedBasicPRM, basic_prm_config, name="BasicPRM")
basic_runner.run_benchmarks(
    [multiRobotBenchList[5]], max_attempts=5,
    fps=50, steps_per_segment=20,
    save_animation=False, animation_dir="animations_multi"
)

#### LazyPRM

In [ ]:
lazy_prm_config = {'initialRoadmapSize': 2000, 'updateRoadmapSize': 100, 'maxIterations': 20} # kNearest wird im MultiRobotPlannerRunner berechnet

basic_runner = MultiRobotPlannerRunner(LazyPRM, lazy_prm_config, name="LazyPRM")
basic_runner.run_benchmarks(
    [multiRobotBenchList[5]], max_attempts=5,
    fps=50, steps_per_segment=20,
    save_animation=False, animation_dir="animations_multi"
)

#### Visibility PRM

In [ ]:
basic_runner = MultiRobotPlannerRunner(VisPRM, {'ntry': 100}, name="VisPRM")
basic_runner.run_benchmarks(
    [multiRobotBenchList[0]], max_attempts=10,
    fps=50, steps_per_segment=20,
    save_animation=False, animation_dir="animations_multi"
)

#### RRT

In [ ]:
basic_runner = MultiRobotPlannerRunner(RRTSimple, {'numberOfGeneratedNodes': 1000, 'testGoalAfterNumberOfNodes': 20}, name="RRTSimple")
basic_runner.run_benchmarks(
    [multiRobotBenchList[0]], max_attempts=5,
    fps=50, steps_per_segment=20,
    save_animation=False, animation_dir="animations_multi"
)

#### Mit dem folgenden Code können Sie alle Multi-Robot Benchmarks gleichzeitig ausführen und die Ergebnisse für die Datenauswertung abspeichern. Die Ausführung aller Benchmarks nimmt etwa **eine Stunde** Zeit in Anspruch. 

#### Zunächst können zudem die Parameter angepasst oder Teilmengen der Benchmarks ausgewählt werden. Für die Planungsverfahren LazyPRM und VisibilityPRM sind zunächst nur Benchmarks mit zwei Robotern vorgesehen.

In [ ]:
# Sollen alle Benchmarks durchgeführt werden, muss zunächst die Variable `skip_benchmark_runs` auf `False` gesetzt werden.
skip_benchmark_runs = True

def run_all_benchmarks():

    # BasicPRM
    config_basic = {
        'radius': 10,
        'numNodes': 300
    }

    # LazyPRM
    config_lazy = {
        'initialRoadmapSize': 2000,
        'updateRoadmapSize': 200,
        'kNearest': 50,
        'maxIterations': 20
    }

    # RRT
    config_rrt = {
        'numberOfGeneratedNodes': 500,
        'testGoalAfterNumberOfNodes': 10
    }

    # VisPRM
    config_vis = {
        'ntry': 40
    }

    # -------------------------------
    # Benchmark-Durchläufe
    # -------------------------------

    bench_range_basic = multiRobotBenchList
    bench_range_lazy = multiRobotBenchList[:5]
    bench_range_vis = multiRobotBenchList[:5]
    bench_range_rrt = multiRobotBenchList

    results = []

    # BasicPRM
    results += run_benchmark_adaptive_multi_try_sampling(
        AdaptedBasicPRM, "BasicPRM", config_basic, bench_range_basic,
        max_attempts=5, scale_factor=1.5, max_scalings=5, 
        params_output_file="results/multi_found_params.json", multi_robot=True
    )

    # LazyPRM
    results += run_benchmark_adaptive_multi_try_sampling(
        LazyPRM, "LazyPRM", config_lazy, bench_range_lazy,
        max_attempts=5, scale_factor=1.5, max_scalings=3, 
        params_output_file="results/multi_found_params.json", multi_robot=True
    )

    # VisPRM
    results += run_benchmark_adaptive_multi_try_sampling(
        VisPRM, "VisPRM", config_vis, bench_range_vis,
        max_attempts=5, scale_factor=1.5, max_scalings=3,
        params_output_file="results/multi_found_params.json", multi_robot=True
    )

    # RRTSimple
    results += run_benchmark_adaptive_multi_try_sampling(
        RRTSimple, "RRTSimple", config_rrt, bench_range_rrt,
        max_attempts=5, scale_factor=1.5, max_scalings=5,
        params_output_file="results/multi_found_params.json", multi_robot=True
    )

    # -------------------------------
    # Ergebnisse speichern
    # -------------------------------

    from IPython.display import display
    import pickle

    with open('results/multi_benchmark_results.pkl', 'wb') as f:
        pickle.dump(results, f)

    df_results = pd.DataFrame(results)
    df_results = df_results[['Planner', 'Benchmark', 'Time [s]', 'Roadmap Size', 'Path Points', 'Path Length']]
    df_results.to_csv('results/multi_benchmark_results.csv', index=False)

    display(df_results)
    
if not skip_benchmark_runs:
    run_all_benchmarks()

#### d. Vergleichen Sie die Planungsverfahren hinsichtlich Suchzeit, Größe Roadmap, Anzahl Punkte im Pfad, Länge des Pfades.
Hierfür werden ausschließlich die .json / .csv / .pkl Dateien benötigt, die nach der Ausführung des vorherigen Codeblocks erstellt wurden. 

Zunächst werden die durch das adaptive Sampling ausgewählten Parameter verglichen:

In [ ]:
visualize_params_custom_layout('results/multi_found_params.json', total_benchmarks=10)

Nun folgt der Vergleich der Suchzeit, Größe der Roadmap, Anzahl Punkte sowie Länge der Pfade:

In [ ]:
# --- Daten laden ---
df_results = pd.read_csv('results/multi_benchmark_results.csv')

# --- Mittelwerte pro Planner berechnen ---
pivot_avg = df_results.groupby("Planner").mean(numeric_only=True).round(3)
pivot_avg = pivot_avg[['Time [s]', 'Roadmap Size', 'Path Points', 'Path Length']]

# --- Min/Max-Highlighting für Tabelle ---
def highlight_min_max(df):
    return df.style.apply(lambda x: [
        'background-color: green' if v == x.min() else
        'background-color: coral' if v == x.max() else ''
        for v in x
    ], axis=0)

print("Durchschnittswerte je Planungsverfahren:")

styled_table = highlight_min_max(pivot_avg).format("{:.3f}")
display(styled_table)


# --- Plots vorbereiten ---
metrics = ['Time [s]', 'Roadmap Size', 'Path Points', 'Path Length']
titles = ['Suchzeit (Sekunden)', 'Größe der Roadmap (Knoten)',
          'Anzahl Punkte im Pfad', 'Pfadlänge (euklidisch)']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

# Einheitliche Farbpalette für alle Planner
unique_planners = df_results['Planner'].unique()
palette = dict(zip(unique_planners, sns.color_palette("Set2", n_colors=len(unique_planners))))

for i, (metric, title) in enumerate(zip(metrics, titles)):
    sns.barplot(
    data=df_results,
    x='Planner',
    y=metric,
    hue='Planner',
    palette=palette,
    dodge=False,
    ax=axes[i]
    )

    axes[i].set_title(title)
    axes[i].set_xlabel("Planungsverfahren")
    axes[i].set_ylabel(title)
    axes[i].grid(True, linestyle='--', alpha=0.6)
    # Legende nur im ersten Plot anzeigen
    if i != 0:
        legend = axes[i].get_legend()
        if legend is not None:
            legend.remove()

# Gemeinsame Legende unten hinzufügen
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(unique_planners), frameon=False)
plt.tight_layout(rect=[0, 0.05, 1, 1])  # Platz für Legende unten
plt.show()


Die Vergleiche nach Suchzeit, Größe der Roadmap, Anzahl Punkte sowie Länge der Pfade werden im Folgenden noch einzeln nach den Benchmarks aufgeschlüsselt.

In [ ]:
# --- CSV laden ---
df_results = pd.read_csv('results/multi_benchmark_results.csv')

metrics = ['Time [s]', 'Roadmap Size', 'Path Points', 'Path Length']

# Planner in fester Reihenfolge (wenn vorhanden)
all_planners = ['BasicPRM', 'LazyPRM', 'RRTSimple', 'VisPRM']
planners_in_data = [p for p in all_planners if p in df_results['Planner'].unique()]

# Benchmarks 1-basiert
benchmarks = sorted(df_results['Benchmark'].unique())
benchmarks_1based = [b + 1 for b in benchmarks]

# Feste Farben für Planner
planner_colors = {
    'BasicPRM': 'green',
    'LazyPRM': 'orange',
    'RRTSimple': 'blue',
    'VisPRM': 'purple'
}

# --- Subplot-Layout ---
n_cols = 2
n_rows = int(np.ceil(len(metrics) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 6 * n_rows))
axes = axes.flatten()

for idx, metric in enumerate(metrics):
    ax = axes[idx]
    x = np.arange(len(benchmarks))
    width = 0.8 / len(planners_in_data)  

    for i, planner in enumerate(planners_in_data):
        values = []
        for benchmark in benchmarks:
            subset = df_results[
                (df_results['Planner'] == planner) & 
                (df_results['Benchmark'] == benchmark)
            ]
            mean_value = subset[metric].mean() if not subset.empty else 0.001
            values.append(mean_value)

        offset = x + (i - len(planners_in_data)/2) * width
        ax.bar(offset, values, width=width, label=planner, color=planner_colors[planner])

    
    ax.set_title(f'{metric} pro Benchmark und Planer', fontsize=11)
    ax.set_xlabel('Benchmark', fontsize=9)
    ax.set_ylabel(metric, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(benchmarks_1based, rotation=45, ha='right', fontsize=8)

    """if metric == 'Time [s]':
        ax.set_yscale('log')
        ax.set_ylim(1e-1, 100)  
        ax.set_ylabel('Time [s] (logarithmisch)', fontsize=9)"""

# Überzählige Achsen löschen
for i in range(len(metrics), len(axes)):
    fig.delaxes(axes[i])

# Gemeinsame Legende unten
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(planners_in_data), fontsize=9, frameon=False)

plt.tight_layout(rect=[0, 0.05, 1, 1])  # Platz unten für Legende
plt.show()


#### e. Erläutern Sie, welche Annahme Sie dabei für die Ausführung der Lösung mit den Robotern treffen müssen.

Bei unserer Planung im hochdimensionalen Konfigurationsraum wird angenommen, dass alle Roboter ihre Bewegung gleichzeitig beginnen und enden. Zudem setzen die geplanten Bahnen eine perfekte Ausführung voraus, was insbesondere bei engen Passagen schwierig wird, da eine solch perfekte Ausführung in der Praxis oft nicht zuverlässig umsetzbar ist. Ebenfalls zeigen unsere Visualisierungen, dass sich die Geschwindigkeit und Position der Roboter entlang ihrer Bahnen teils abrupt ändert. Für die Ausführung ohne vorgelagertes Glätten muss also angenommen werden, dass die Roboter solche ruckartigen Änderungen in der Bewegungsbahn ohne Verzögerung umsetzen können.